# Analyse exploratoire — CropYield Prediction

Second jeu fourni. Il ne décrit pas des parcelles mais des **pays**, année par année :
rendement par culture, température, pluviométrie et usage de pesticides, de source FAO
et Banque mondiale.

Le sujet fournit cinq fichiers : quatre sources brutes (`yield`, `temp`, `rainfall`,
`pesticides`) et un fichier `yield_df.csv` déjà fusionné. Ce notebook examine les quatre
sources **et audite le fichier fusionné**. Cet audit est décisif : il montre que
`yield_df.csv` est inutilisable en l'état, ce qui justifie la reconstruction menée au
notebook 04.

# Imports

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

from src import config, utils
from src.data import crop_yield_prediction as cyp
from src.data import geo

utils.style()
np.random.seed(config.SEED)

# Lecture des sources

In [2]:
brut = cyp.charge_sources()
for nom, df in brut.items():
    print(f"{nom:12s} {df.shape[0]:6,} lignes x {df.shape[1]} colonnes | "
          f"{list(df.columns)}")

rendement    56,717 lignes x 12 colonnes | ['Domain Code', 'Domain', 'Area Code', 'Area', 'Element Code', 'Element', 'Item Code', 'Item', 'Year Code', 'Year', 'Unit', 'Value']
temperature  71,311 lignes x 3 colonnes | ['year', 'country', 'avg_temp']
pluie         6,727 lignes x 3 colonnes | ['Area', 'Year', 'average_rain_fall_mm_per_year']
pesticides    4,349 lignes x 7 colonnes | ['Domain', 'Area', 'Element', 'Item', 'Year', 'Unit', 'Value']


## yield.csv

In [3]:
rendement = brut["rendement"]
print("Element :", rendement["Element"].unique(), "| Unit :", rendement["Unit"].unique())
print("cultures :", sorted(rendement["Item"].unique()))
print(f"années   : {rendement['Year'].min()} - {rendement['Year'].max()}")
utils.profil(rendement, "yield")

Element : <StringArray>
['Yield']
Length: 1, dtype: str | Unit : <StringArray>
['hg/ha']
Length: 1, dtype: str
cultures : ['Cassava', 'Maize', 'Plantains and others', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans', 'Sweet potatoes', 'Wheat', 'Yams']
années   : 1961 - 2016


,type,manquantes,manquantes_%,modalites,exemple
colonne yield,,,,,
Domain Code,str,0,0.0,1,QC
Domain,str,0,0.0,1,Crops
Area Code,int64,0,0.0,212,2
Area,str,0,0.0,212,Afghanistan
Element Code,int64,0,0.0,1,5419
Element,str,0,0.0,1,Yield
Item Code,int64,0,0.0,10,56
Item,str,0,0.0,10,Maize
Year Code,int64,0,0.0,56,1961


**Observations :**

- Une seule grandeur (`Yield`) dans une seule unité (**hg/ha**, hectogrammes par
  hectare) : à convertir en t/ha pour être comparable au premier jeu (division par
  10 000).
- **Dix cultures**, dont quatre seulement recoupent les six du jeu simulé.
- La période couverte, 1961-2016, est bien plus large que celle des autres sources.

## temp.csv

In [4]:
temperature = brut["temperature"]
print(f"doublons stricts : {temperature.duplicated().sum():,}")
print(f"avg_temp manquantes : {temperature['avg_temp'].isna().sum():,}")
releves = temperature.drop_duplicates().groupby(["country", "year"]).size()
print(f"\nrelevés par couple pays-année : min {releves.min()}, "
      f"médiane {int(releves.median())}, max {releves.max()}")
print("\nexemple — Argentine 1990 :")
temperature[(temperature["country"] == "Argentina") & (temperature["year"] == 1990)]

doublons stricts : 6,958
avg_temp manquantes : 2,547

relevés par couple pays-année : min 1, médiane 1, max 41

exemple — Argentine 1990 :


,year,country,avg_temp
17130,1990,Argentina,17.46
55988,1990,Argentina,17.67


**Observations — le point le plus important de ce notebook :**

- Après retrait des doublons stricts, un même couple pays-année peut porter **jusqu'à 41
  relevés** de température distincts.
- Joindre ce fichier sans agréger au préalable **démultiplie les lignes** : c'est un
  fan-out, et c'est exactement l'erreur commise lors de la construction de
  `yield_df.csv` (voir l'audit en fin de notebook).
- 2 547 relevés sont vides.
- Décision pour le notebook 04 : retirer les doublons stricts, écarter les relevés vides,
  puis **moyenner par pays-année**. Aucune métadonnée ne permettant de départager les
  stations, la moyenne non pondérée est le choix le plus neutre.

## rainfall.csv

In [5]:
pluie = brut["pluie"]
converti = pd.to_numeric(pluie["average_rain_fall_mm_per_year"], errors="coerce")
print(f"type lu : {pluie['average_rain_fall_mm_per_year'].dtype}")
print(f"valeurs non numériques ou vides : {converti.isna().sum()} "
      f"({converti.isna().mean():.1%})")
print(f"valeurs distinctes : {converti.nunique()} pour {pluie['Area'].nunique()} pays")
print()
# Does rainfall vary over time for a given country?
variation = (pluie.assign(mm=converti).dropna(subset=["mm"])
             .groupby("Area")["mm"].nunique())
print(f"pays dont la pluviométrie varie selon l'année : {(variation > 1).sum()} "
      f"sur {len(variation)}")

type lu : str
valeurs non numériques ou vides : 780 (11.6%)
valeurs distinctes : 172 pour 217 pays

pays dont la pluviométrie varie selon l'année : 4 sur 192


**Observations :**

- La colonne est lue en **texte** : des cellules vides empêchent la conversion. Elles
  deviennent `NaN` et **ne seront pas imputées** — il s'agit d'une normale climatique par
  pays, qu'aucune valeur voisine ne permet d'inventer.
- **La pluviométrie est quasi constante dans le temps** : elle ne varie selon l'année que pour 4 pays sur 192. Ce n'est pas une
  mesure annuelle mais une **normale climatique**. Conséquence directe : cette variable ne
  peut pas expliquer une variation de rendement d'une année sur l'autre, seulement un
  écart entre pays.
- L'en-tête de la colonne pays commence par une espace, et certaines valeurs portent une
  espace finale (`"South Korea "`).

## pesticides.csv

In [6]:
pesticides = brut["pesticides"]
print("Element :", pesticides["Element"].unique(), "| Item :", pesticides["Item"].unique())
print("Unit :", pesticides["Unit"].unique())
print(f"doublons sur (Area, Year) : {pesticides.duplicated(['Area', 'Year']).sum()}")
print(f"années : {pesticides['Year'].min()} - {pesticides['Year'].max()}")
pesticides["Value"].describe().round(1)

Element : <StringArray>
['Use']
Length: 1, dtype: str | Item : <StringArray>
['Pesticides (total)']
Length: 1, dtype: str
Unit : <StringArray>
['tonnes of active ingredients']
Length: 1, dtype: str
doublons sur (Area, Year) : 0
années : 1990 - 2016


count       4349.0
mean       20303.3
std       117736.2
min            0.0
25%           93.0
50%         1137.6
75%         7869.0
max      1807000.0
Name: Value, dtype: float64

**Observations :**

- Une seule ligne par pays et par année : aucune agrégation nécessaire.
- Un **total national en tonnes de matière active**, toutes cultures confondues. Ce n'est
  donc pas une pratique de parcelle : un agriculteur ne peut pas renseigner le tonnage de
  son pays. La variable servira de **proxy d'intensification**, pas d'entrée utilisateur.
- La distribution est très étalée (facteur 10⁷ entre les extrêmes), à l'image des écarts
  de taille entre pays.

# Période commune aux sources

In [7]:
periodes = pd.DataFrame({
    nom: {"début": int(df[col].min()), "fin": int(df[col].max())}
    for nom, (df, col) in {
        "rendement": (brut["rendement"], "Year"),
        "temperature": (brut["temperature"], "year"),
        "pluie": (brut["pluie"], "Year"),
        "pesticides": (brut["pesticides"], "Year"),
    }.items()
}).T
periodes["intersection"] = f"{config.ANNEE_DEBUT}-{config.ANNEE_FIN}"
periodes

,début,fin,intersection
rendement,1961,2016,1990-2013
temperature,1743,2013,1990-2013
pluie,1985,2017,1990-2013
pesticides,1990,2016,1990-2013


**Observations :** l'intersection des quatre sources est **1990-2013**, imposée par
`pesticides.csv`. C'est la période retenue pour la reconstruction.

# Couverture géographique : trois conventions de nommage

Les quatre fichiers ne nomment pas les pays de la même façon. C'est le second défaut
structurel de ce jeu de données, et il conditionne toute la fusion.

In [8]:
noms = {
    "rendement": set(brut["rendement"]["Area"].astype(str).unique()),
    "temperature": set(brut["temperature"]["country"].astype(str).unique()),
    "pluie": set(brut["pluie"]["Area"].astype(str).unique()),
    "pesticides": set(brut["pesticides"]["Area"].astype(str).unique()),
}
for nom, ensemble in noms.items():
    print(f"{nom:12s} {len(ensemble):3d} noms distincts")

union = set.union(*noms.values())
intersection_litterale = set.intersection(*noms.values())
print(f"\nunion                   : {len(union)} noms")
print(f"intersection littérale  : {len(intersection_litterale)} pays")

rendement    212 noms distincts
temperature  137 noms distincts
pluie        217 noms distincts
pesticides   168 noms distincts

union                   : 270 noms
intersection littérale  : 101 pays


In [9]:
# How does each source name the same major countries?
exemples = ["United States", "Russia", "Vietnam", "South Korea", "Czech", "China",
            "Venezuela", "Ivoire|Cote"]
import re
comparaison = []
for motif in exemples:
    ligne = {"motif": motif}
    for nom, ensemble in noms.items():
        trouve = sorted(x for x in ensemble if re.search(motif, x, re.I))
        ligne[nom] = " / ".join(trouve) if trouve else "—"
    comparaison.append(ligne)
pd.DataFrame(comparaison).set_index("motif")

,rendement,temperature,pluie,pesticides
motif,,,,
United States,United States of America,United States,United States,United States of America
Russia,Russian Federation,Russia,Russia,Russian Federation
Vietnam,—,Vietnam,Vietnam,—
South Korea,—,South Korea,South Korea,—
Czech,Czechia / Czechoslovakia,Czech Republic,Czech Republic,Czechia
China,"China / China, Hong Kong SAR / China, Taiwan P...",China,"China / Hong Kong SAR, China / Macao SAR, China","China, Hong Kong SAR / China, Macao SAR / Chin..."
Venezuela,Venezuela (Bolivarian Republic of),Venezuela,"Venezuela, RB",Venezuela (Bolivarian Republic of)
Ivoire|Cote,Côte d'Ivoire,Côte D'Ivoire,Cote d'Ivoire,Côte d'Ivoire


**Observations :**

| Source | Convention | Exemples |
|---|---|---|
| `yield`, `pesticides` | forme longue FAO | `United States of America`, `Viet Nam`, `Republic of Korea`, `China, mainland` |
| `temp` | noms courants | `United States`, `Vietnam`, `South Korea`, `Czech Republic` |
| `rainfall` | style Banque mondiale | `Venezuela, RB`, `Hong Kong SAR, China`, `"South Korea "` |

- **L'intersection littérale ne retient que 101 pays** sur 270 noms distincts.
- Les pays perdus ne sont pas marginaux : **États-Unis, Chine, Russie, Iran, Viêt Nam,
  Corée du Sud, Tanzanie, Tchéquie, Turquie** — parmi les premiers producteurs mondiaux.
- La solution est d'apparier par **code ISO 3166-1 alpha-3**, ce que fait
  `src/data/geo.py`.

In [10]:
# Gain brought by ISO3 matching.
correspondances = geo.vers_iso3(union)
codes = {nom: {correspondances[x] for x in ensemble if correspondances.get(x)}
         for nom, ensemble in noms.items()}
intersection_iso3 = set.intersection(*codes.values())

print(f"intersection littérale : {len(intersection_litterale):3d} pays")
print(f"intersection ISO3      : {len(intersection_iso3):3d} pays "
      f"(+{len(intersection_iso3) - len(intersection_litterale)})")
print()
for code in ["USA", "CHN", "RUS", "IRN", "VNM", "KOR", "TZA", "CZE", "TUR"]:
    print(f"  {code} : {'présent' if code in intersection_iso3 else 'ABSENT'}")
print()
sans_code = sorted(x for x in union if not correspondances.get(x))
print(f"noms sans code ISO3 ({len(sans_code)}) : {sans_code}")

intersection littérale : 101 pays
intersection ISO3      : 120 pays (+19)

  USA : présent
  CHN : présent
  RUS : présent
  IRN : présent
  VNM : présent
  KOR : présent
  TZA : présent
  CZE : présent
  TUR : présent

noms sans code ISO3 (8) : ['Belgium-Luxembourg', 'Channel Islands', 'Czechoslovakia', 'Pacific Islands Trust Territory', 'Serbia and Montenegro', 'Sudan (former)', 'USSR', 'Yugoslav SFR']


**Observations :**

- L'appariement ISO3 porte la couverture de **101 à 120 pays**, et récupère **tous** les
  grands producteurs.
- Les huit noms sans code sont exclusivement des **entités historiques disparues** (URSS,
  Tchécoslovaquie, Yougoslavie, Serbie-et-Monténégro, Belgique-Luxembourg, Soudan ex) ou
  des agrégats non géographiques. Les écarter est un choix documenté, pas un oubli.

# Audit de `yield_df.csv`

Le fichier pré-fusionné fourni avec le sujet. On vérifie s'il est exploitable tel quel.

In [11]:
prefusionne = pd.read_csv(config.SOURCE_PREFUSIONNEE, index_col=0)
print(f"{prefusionne.shape[0]:,} lignes x {prefusionne.shape[1]} colonnes")
print(f"pays : {prefusionne['Area'].nunique()} | cultures : {prefusionne['Item'].nunique()}")

cle = ["Area", "Item", "Year"]
masque = prefusionne.duplicated(cle, keep=False)
groupes = prefusionne.loc[masque].groupby(cle).ngroups
print(f"\nlignes impliquées dans une clé dupliquée : {masque.sum():,} "
      f"({masque.mean():.1%} du fichier)")
print(f"groupes de clés dupliquées : {groupes:,}")
print(f"lignes excédentaires : {masque.sum() - groupes:,}")

28,242 lignes x 7 colonnes


pays : 101 | cultures : 10

lignes impliquées dans une clé dupliquée : 18,949 (67.1% du fichier)
groupes de clés dupliquées : 3,837
lignes excédentaires : 15,112


In [12]:
# What differs within a duplicated group?
colonnes_valeurs = ["hg/ha_yield", "average_rain_fall_mm_per_year",
                    "pesticides_tonnes", "avg_temp"]
groupe = prefusionne.loc[masque].groupby(cle)
varie = pd.DataFrame({
    "groupes où la valeur diffère": {
        col: int((groupe[col].nunique() > 1).sum()) for col in colonnes_valeurs
    }
})
varie["part"] = (varie.iloc[:, 0] / groupes).map("{:.1%}".format)
print(varie.to_string())
print()
print("exemple — Argentine / Maïs / 1990 :")
prefusionne[(prefusionne["Area"] == "Argentina") & (prefusionne["Item"] == "Maize")
            & (prefusionne["Year"] == 1990)]

                               groupes où la valeur diffère   part
hg/ha_yield                                               0   0.0%
average_rain_fall_mm_per_year                             0   0.0%
pesticides_tonnes                                         0   0.0%
avg_temp                                               3690  96.2%

exemple — Argentine / Maïs / 1990 :


,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
379,Argentina,Maize,1990,34608,591.0,26156.0,17.46
380,Argentina,Maize,1990,34608,591.0,26156.0,17.67


**Observations — `yield_df.csv` est inexploitable en l'état :**

1. **54 % du fichier est redondant.** 3 837 combinaisons pays × culture × année se
   répètent, soit 15 112 lignes excédentaires.
2. **La cause est identifiée** : seule `avg_temp` diffère d'une occurrence à l'autre,
   tandis que rendement, pluie et pesticides sont identiques. C'est la signature du
   fan-out de `temp.csv` décrit plus haut — le fichier a été construit sans agréger les
   relevés de température au préalable.
3. **Il ne contient que 101 pays**, soit exactement l'intersection littérale des noms.
   Il a donc été assemblé par jointure sur le nom des pays, perdant silencieusement les
   États-Unis, la Chine et la Russie.

> Une analyse menée sur ce fichier accorderait un poids jusqu'à 41 fois supérieur à
> certains pays-années, tout en ignorant les premiers producteurs mondiaux.

**Décision : ne pas utiliser `yield_df.csv`.** Le jeu est reconstruit depuis les quatre
sources brutes au notebook 04, avec appariement ISO3 et contrôles anti-fan-out. On
corrige ainsi la cause plutôt que le symptôme.

# Conclusion

## Ce que contiennent les sources
Quatre fichiers à des granularités différentes : le rendement par **pays × année ×
culture** (10 cultures, hg/ha), la température par **pays × année** (plusieurs relevés),
la pluie par **pays** (normale climatique constante dans le temps) et les pesticides par
**pays × année** (total national).

## Ce qui est exploitable
- Dix cultures nettement différenciées — à confronter au jeu simulé (notebook 05).
- Une période commune de **1990 à 2013**, imposée par `pesticides.csv`.
- **120 pays** après appariement ISO3, contre 101 par nom littéral.

## Les points de qualité à traiter
- `temp.csv` : jusqu'à **41 relevés** par pays-année → agrégation obligatoire avant toute
  jointure, sous peine de fan-out.
- `rainfall.csv` : colonne lue en texte, 780 valeurs vides, en-tête et valeurs avec des
  espaces parasites. **Le fichier ne contient par ailleurs aucune ligne pour 2003**, ce qui
  prive cette année de pluviométrie à la jointure.
- Trois conventions de nommage des pays → appariement par code ISO3.
- `yield.csv` : `China` est un agrégat incluant Taïwan, Hong Kong et Macao, distinct de
  `China, mainland`.
- **`yield_df.csv` écarté** : 54 % de redondance et 101 pays seulement.

## Décisions à confirmer plus tard
- Comment agréger les relevés de température : moyenne simple, ou pondération ? Faute de
  métadonnée sur les stations, la moyenne simple est retenue par défaut.
- Que faire des valeurs manquantes de pluie et de pesticides après fusion : les conserver
  en `NaN`, ou restreindre l'analyse aux lignes complètes ? *(Tranché au notebook 04 : la
  pluie étant une constante nationale, une valeur par pays est retenue et diffusée sur
  toutes ses années.)*
- La pluviométrie étant une constante par pays, faut-il la traiter comme une variable
  climatique ou comme un simple identifiant de pays ? *(Question rouverte à l'étape 2 :
  elle y dégrade la généralisation du modèle de recommandation.)*

## Hypothèses à vérifier plus tard
- Les dix cultures devraient présenter des rendements nettement distincts, ce qui
  fournirait l'effet culture absent du jeu simulé.
- Les écarts entre cultures risquent de refléter surtout **où** elles sont cultivées,
  plutôt que leurs qualités agronomiques propres : à tester au notebook 05.